### Gold layer: season_simulations

Monte Carlo (10,000 runs) over the remaining matchdays of the active season,
starting from the real current table + each team's latest ratings.
No freeze/is_provisional (this is a point-in-time projection).
In-memory only until validated against the real table.

In [1]:
import sys
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow as pa

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from utils.parquet_io import read_league_all_seasons, read_partition, write_partition
from utils.dixon_coles import score_matrix
from utils.schema import SEASON_SIMULATIONS_SCHEMA

LAKE_ROOT = PROJECT_ROOT / "data"
LEAGUES = ["laliga", "primeira_liga"]
ACTIVE_SEASON = "2026-2027"
N_SIMS = 10_000

silver = {lg: read_league_all_seasons(LAKE_ROOT, layer="silver", league=lg).to_pandas()
          for lg in LEAGUES}
ratings = {lg: read_league_all_seasons(LAKE_ROOT, layer="gold", league=lg,
                                       filename="team_ratings.parquet").to_pandas()
           for lg in LEAGUES}

for lg in LEAGUES:
    print(lg, "silver:", silver[lg].shape, "| ratings:", ratings[lg].shape)

laliga silver: (1520, 19) | ratings: (2440, 17)
primeira_liga silver: (1224, 19) | ratings: (1980, 17)


In [2]:
LEAGUE_SLOTS = {
    "laliga": {
        "n_teams": 20,
        "champions_league": range(1, 5),     # positions 1-4
        "relegation": range(18, 21),         # positions 18-20
    },
    "primeira_liga": {
        "n_teams": 18,
        "champions_league": range(1, 3),         # positions 1-2, direct group stage
        "champions_qualifying": (3,),            # position 3, qualifying round
        "relegation": range(17, 19),             # positions 17-18, direct relegation
        "relegation_playoff": (16,),             # position 16, relegation playoff
    },
}

In [3]:
def get_current_standings(matches, season):
    played = matches[(matches["season"] == season) & (matches["status"] == "FINISHED")]
    teams = pd.unique(played[["home_team_key", "away_team_key"]].values.ravel())
    points = {t: 0 for t in teams}
    goal_diff = {t: 0 for t in teams}
    for r in played.itertuples():
        goal_diff[r.home_team_key] += r.goal_difference
        goal_diff[r.away_team_key] -= r.goal_difference
        if r.result == "H":
            points[r.home_team_key] += 3
        elif r.result == "A":
            points[r.away_team_key] += 3
        else:
            points[r.home_team_key] += 1
            points[r.away_team_key] += 1
    return points, goal_diff


def get_remaining_fixtures(matches, season):
    rem = matches[(matches["season"] == season) & (matches["status"] != "FINISHED")]
    return rem[["matchday", "match_date", "home_team_key", "away_team_key"]] \
        .sort_values(["matchday", "match_date"]).reset_index(drop=True)


def get_latest_ratings(rat, season):
    s = rat[rat["season"] == season]
    last_md = int(s["matchday"].max())
    snap = s[s["matchday"] == last_md].set_index("team_key")[["attack", "defense", "mu", "gamma", "rho"]]
    return snap, last_md


standings, fixtures, ratings_snap, as_of_md = {}, {}, {}, {}
for lg in LEAGUES:
    standings[lg] = get_current_standings(silver[lg], ACTIVE_SEASON)
    fixtures[lg] = get_remaining_fixtures(silver[lg], ACTIVE_SEASON)
    ratings_snap[lg], as_of_md[lg] = get_latest_ratings(ratings[lg], ACTIVE_SEASON)
    print(f"{lg}: {len(fixtures[lg])} partidos restantes | ratings as_of matchday {as_of_md[lg]}")

laliga: 311 partidos restantes | ratings as_of matchday 8
primeira_liga: 244 partidos restantes | ratings as_of matchday 8


### Validation: computed current table vs. real table

Compare this output against the Wikipedia table (matchday 7, 2026-09-20)
before continuing. If any team/points don't match, stop here.

In [4]:
for lg in LEAGUES:
    pts, gd = standings[lg]
    tbl = pd.DataFrame({"team_key": list(pts), "points": list(pts.values()),
                         "goal_diff": [gd[t] for t in pts]})
    tbl = tbl.sort_values(["points", "goal_diff"], ascending=False).reset_index(drop=True)
    tbl.index += 1
    print(f"=== {lg} ===")
    print(tbl.head(10))
    print()

=== laliga ===
                   team_key  points  goal_diff
1              fc_barcelona      21         24
2   club_atlético_de_madrid      16          9
3       real_betis_balompié      16          2
4            real_madrid_cf      15         10
5                sevilla_fc      13          1
6          deportivo_alavés      11          5
7              rc_deportivo      10          2
8   real_sociedad_de_fútbol      10         -4
9             villarreal_cf       8          1
10            athletic_club       8          1

=== primeira_liga ===
                      team_key  points  goal_diff
1                     fc_porto      21         15
2       sport_lisboa_e_benfica      16         15
3   sporting_clube_de_portugal      15          9
4               cd_santa_clara      15          7
5                    fc_arouca      11          3
6      sporting_clube_de_braga      11          2
7           académico_de_viseu      11          0
8        cf_estrela_da_amadora      10       

In [5]:
def simulate_season(fixtures, ratings_snap, start_points, start_gd, rng, max_goals=10):
    mu, gamma, rho = ratings_snap[["mu", "gamma", "rho"]].iloc[0]
    points = dict(start_points)
    goal_diff = dict(start_gd)
    for f in fixtures.itertuples():
        h_att, h_def = ratings_snap.loc[f.home_team_key, ["attack", "defense"]]
        a_att, a_def = ratings_snap.loc[f.away_team_key, ["attack", "defense"]]
        lh = np.exp(mu + gamma + h_att - a_def)
        la = np.exp(mu + a_att - h_def)
        matrix = score_matrix(lh, la, rho, max_goals=max_goals)
        flat = matrix.flatten()
        idx = rng.choice(len(flat), p=flat)
        gh, ga = divmod(idx, matrix.shape[1])
        diff = gh - ga
        goal_diff[f.home_team_key] += diff
        goal_diff[f.away_team_key] -= diff
        if gh > ga:
            points[f.home_team_key] += 3
        elif gh < ga:
            points[f.away_team_key] += 3
        else:
            points[f.home_team_key] += 1
            points[f.away_team_key] += 1
    table = pd.DataFrame({"team_key": list(points), "points": list(points.values()),
                           "goal_diff": [goal_diff[t] for t in points]})
    table = table.sort_values(["points", "goal_diff"], ascending=False).reset_index(drop=True)
    table["position"] = table.index + 1
    return table


def run_monte_carlo(fixtures, ratings_snap, start_points, start_gd, slots, n_sims=N_SIMS, seed=42):
    rng = np.random.default_rng(seed)
    teams = list(start_points.keys())
    acc = {t: {"champion": 0, "cl": 0, "cl_q": 0, "releg": 0, "releg_po": 0, "pos_sum": 0} for t in teams}
    for _ in range(n_sims):
        table = simulate_season(fixtures, ratings_snap, start_points, start_gd, rng)
        for row in table.itertuples():
            c = acc[row.team_key]
            c["pos_sum"] += row.position
            c["champion"] += row.position == 1
            c["cl"] += row.position in slots["champions_league"]
            c["cl_q"] += row.position in slots.get("champions_qualifying", ())
            c["releg"] += row.position in slots["relegation"]
            c["releg_po"] += row.position in slots.get("relegation_playoff", ())
    out = pd.DataFrame([
        {"team_key": t,
         "prob_champion": c["champion"] / n_sims,
         "prob_champions_league": c["cl"] / n_sims,
         "prob_champions_qualifying": c["cl_q"] / n_sims,
         "prob_relegation": c["releg"] / n_sims,
         "prob_relegation_playoff": c["releg_po"] / n_sims,
         "avg_position": c["pos_sum"] / n_sims}
        for t, c in acc.items()
    ])
    return out.sort_values("avg_position").reset_index(drop=True)

In [6]:
sims = {}
for lg in LEAGUES:
    pts, gd = standings[lg]
    result = run_monte_carlo(fixtures[lg], ratings_snap[lg], pts, gd, LEAGUE_SLOTS[lg])
    result["league"] = lg
    result["season"] = ACTIVE_SEASON
    result["as_of_matchday"] = as_of_md[lg]
    result["n_simulations"] = N_SIMS
    result["simulated_at"] = datetime.now(timezone.utc)
    sims[lg] = result
    print(f"=== {lg} ===")
    print(result.head(10))
    print()

=== laliga ===
                  team_key  prob_champion  prob_champions_league  \
0             fc_barcelona         0.8884                 0.9998   
1           real_madrid_cf         0.0922                 0.9873   
2  club_atlético_de_madrid         0.0169                 0.8938   
3      real_betis_balompié         0.0022                 0.5628   
4            villarreal_cf         0.0003                 0.2556   
5            athletic_club         0.0000                 0.0910   
6  real_sociedad_de_fútbol         0.0000                 0.0534   
7               sevilla_fc         0.0000                 0.0486   
8         deportivo_alavés         0.0000                 0.0284   
9         rc_celta_de_vigo         0.0000                 0.0250   

   prob_champions_qualifying  prob_relegation  prob_relegation_playoff  \
0                        0.0           0.0000                      0.0   
1                        0.0           0.0000                      0.0   
2             

### Validation: probability sums

prob_champion should sum to ~1.0 per league; prob_champions_league should
sum to ~that league's slots (4 / 2); prob_relegation ~relegation slots (3 / 2);
qualifying/playoff (Primeira Liga only) ~1.0 each.

In [7]:
for lg in LEAGUES:
    s = sims[lg]
    print(f"=== {lg} ===")
    print("prob_champion sum:", round(s["prob_champion"].sum(), 3))
    print("prob_champions_league sum:", round(s["prob_champions_league"].sum(), 3))
    print("prob_champions_qualifying sum:", round(s["prob_champions_qualifying"].sum(), 3))
    print("prob_relegation sum:", round(s["prob_relegation"].sum(), 3))
    print("prob_relegation_playoff sum:", round(s["prob_relegation_playoff"].sum(), 3))
    print()

=== laliga ===
prob_champion sum: 1.0
prob_champions_league sum: 4.0
prob_champions_qualifying sum: 0.0
prob_relegation sum: 3.0
prob_relegation_playoff sum: 0.0

=== primeira_liga ===
prob_champion sum: 1.0
prob_champions_league sum: 2.0
prob_champions_qualifying sum: 1.0
prob_relegation sum: 2.0
prob_relegation_playoff sum: 1.0



In [8]:
COLS = [f.name for f in SEASON_SIMULATIONS_SCHEMA]

for lg in LEAGUES:
    part = sims[lg][COLS]
    path = LAKE_ROOT / "gold" / "season_simulations" / lg / f"season={ACTIVE_SEASON}" / "season_simulations.parquet"
    table = pa.Table.from_pandas(part, schema=SEASON_SIMULATIONS_SCHEMA, preserve_index=False)
    write_partition(table, path)
    print(f"{lg}: {len(part)} rows -> {path.relative_to(PROJECT_ROOT)}")

laliga: 20 rows -> data\gold\season_simulations\laliga\season=2026-2027\season_simulations.parquet
primeira_liga: 18 rows -> data\gold\season_simulations\primeira_liga\season=2026-2027\season_simulations.parquet
